In [2]:
import pandas as pd
import numpy as np

In [3]:
import os

PROJECT_ROOT = r"C:\Users\nicol\OneDrive\Bureau\Finance"
os.chdir(PROJECT_ROOT)

print("Current working directory:", os.getcwd())

Current working directory: C:\Users\nicol\OneDrive\Bureau\Finance


In [4]:
df_features = pd.read_csv("data/processed/features.csv", index_col=0, parse_dates=True)
df_features.head(10)

,Open,High,Low,Close,Volume,log_return,vol_5,vol_10,vol_21,vol_ewma,ATR,RSI,MACD,MACD_signal,log_volume,volume_change,target_vol
Date,,,,,,,,,,,,,,,,,
2015-02-03 00:00:00-05:00,2022.709961,2050.300049,2022.709961,2050.030029,4615900000,0.014336,0.014115,0.011876,0.011401,0.011663,30.628584,54.949410,-0.174383,-0.104848,22.252773,0.141132,0.010659
2015-02-04 00:00:00-05:00,2048.860107,2054.739990,2036.719971,2041.510010,4141920000,-0.004165,0.011980,0.011928,0.010659,0.011154,29.445007,55.609976,0.788912,0.075233,22.144425,-0.108347,0.010632
2015-02-05 00:00:00-05:00,2043.449951,2063.550049,2043.449951,2062.520020,3821990000,0.010239,0.012071,0.011342,0.010632,0.010990,28.885010,62.843383,2.954017,0.654408,22.064037,-0.080388,0.010417
2015-02-06 00:00:00-05:00,2062.280029,2072.399902,2049.969971,2055.469971,4232970000,-0.003424,0.009040,0.011249,0.010417,0.010524,28.177150,57.146245,4.116070,1.350026,22.166170,0.102133,0.009699
2015-02-09 00:00:00-05:00,2053.469971,2056.159912,2041.880005,2046.739990,3549540000,-0.004256,0.009016,0.011295,0.009699,0.010109,27.450718,54.691067,4.364544,1.955216,21.990084,-0.176086,0.009796
2015-02-10 00:00:00-05:00,2049.379883,2070.860107,2048.620117,2068.590088,3669850000,0.010619,0.007882,0.010778,0.009796,0.010085,27.298584,56.750971,6.041058,2.774861,22.023417,0.033333,0.009595
2015-02-11 00:00:00-05:00,2068.550049,2073.479980,2057.989990,2068.530029,3596860000,-0.000029,0.007295,0.009374,0.009595,0.009583,25.673575,51.124890,7.272450,3.676559,22.003327,-0.020090,0.009738
2015-02-12 00:00:00-05:00,2069.979980,2088.530029,2069.979980,2088.479980,3788350000,0.009598,0.007132,0.009382,0.009738,0.009442,26.201442,57.398263,9.574355,4.858404,22.055196,0.051869,0.009604
2015-02-13 00:00:00-05:00,2088.780029,2097.030029,2086.699951,2096.989990,3527450000,0.004066,0.006316,0.007425,0.009604,0.008988,25.750009,57.948180,11.872490,6.263396,21.983841,-0.071355,0.009252


In [5]:
df_features.replace([np.inf, -np.inf], np.nan, inplace=True)
df_features.dropna(inplace=True)

In [6]:
FEATURES = [
    "vol_5", "vol_10", "vol_21", "vol_ewma",
    "ATR", "RSI", "MACD", "MACD_signal",
    "log_volume", "volume_change"
]

TARGET = "target_vol"

In [7]:
split = int(len(df_features) * 0.8)

train = df_features.iloc[:split]
test = df_features.iloc[split:]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

In [8]:
from src.models.ml import train_random_forest

rf_model = train_random_forest(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [9]:
from src.models.ml import train_gb

gb_model = train_gb(X_train, y_train)
y_pred_gb = gb_model.predict(X_test)

In [10]:
from src.evaluation.metrics import evaluate_volatility

results_ml = {
    "RandomForest": evaluate_volatility(y_test, y_pred_rf),
    "GradientBoosting": evaluate_volatility(y_test, y_pred_gb)
}

results_ml_df = pd.DataFrame(results_ml).T
print(results_ml_df)

                           MSE       MAE     QLIKE
RandomForest      4.685665e-07  0.000442 -8.066527
GradientBoosting  1.000236e-06  0.000634 -8.058304


In [11]:
import pandas as pd

feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print(feature_importance)

vol_ewma         0.534670
vol_21           0.452737
MACD_signal      0.007362
vol_10           0.004202
ATR              0.000658
RSI              0.000167
log_volume       0.000087
vol_5            0.000057
MACD             0.000052
volume_change    0.000009
dtype: float64


In [13]:
import joblib

joblib.dump(rf_model, "src/models/rf_model.pkl")

['src/models/rf_model.pkl']